In [1]:
!pip install numpy matplotlib torch diffusers scipy

In [2]:
!pip install torchvision

In [3]:
!pip install timm

In [4]:
# Parses the rollout data from demo.py and saves them as numpy arrays of states, actions, and pot handle positions (the conditional vector data)

import numpy as np
import csv
import matplotlib.pyplot as plt
import pickle as pkl
import torch
from scipy.spatial.transform import Rotation as R
from transform_utils import quat_to_rot6d, rotvec_to_rot6d, rot6d_to_quat
from diffusers import AutoencoderKL
import os
from net import TimMResNet18Encoder


/home/haoyu/.conda/envs/mult_diff/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Initialize ViT encoder for image processing
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# vit_encoder = VisionTransformerEncoder(latent_dim=128)
# vit_encoder.to(device).eval()

In [6]:
# Example usage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = TimMResNet18Encoder(pretrained=True, latent_dim=128).to(device)
encoder.eval()

TimMResNet18Encoder(
  (backbone): FeatureListNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act1): ReLU(inplace=True)
        (aa): Identity()
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=

In [7]:
# with open("rollouts_pot/rollout_seed0_mode2.pkl", "rb") as f:
#     rollout = pkl.load(f)
#     obs = rollout["observations"]
#     actions = np.array(rollout["actions"])
#     robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
#     robot0_eef_quat = np.array([o["robot0_eef_quat"] for o in obs])
#     robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
#     robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
#     robot1_eef_quat = np.array([o["robot1_eef_quat"] for o in obs])
#     robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

#     repeats_needed = 250 - actions.shape[0]

#     repeated_last = np.tile(actions[-1], (repeats_needed, 1))
#     actions = np.vstack([actions, repeated_last])

#     repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
#     robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
#     state = robot0_eef_pos

#     repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
#     robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
#     robot0_eef_rotvec = R.from_quat(robot0_eef_quat).as_rotvec()
#     state = np.hstack([state, robot0_eef_rotvec])


#     repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
#     robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
#     robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
#     state = np.hstack([state, robot0_gripper_pos])

#     repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
#     robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
#     state = np.hstack([state, robot1_eef_pos])

#     repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
#     robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
#     robot1_eef_rotvec = R.from_quat(robot1_eef_quat).as_rotvec()
#     state = np.hstack([state, robot1_eef_rotvec])

#     repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
#     robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
#     robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
#     state = np.hstack([state, robot1_gripper_pos])

# print(np.shape(state))
# print(np.shape(actions))

In [8]:
# Use PRE-TRAINED ResNet for image encoding (NO training needed!)
# import torchvision.models as models
# import torch.nn as nn

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # Load pre-trained ResNet18
# resnet = models.resnet18(pretrained=True)
# encoder_backbone = nn.Sequential(*list(resnet.children())[:-1])  # Remove final FC layer
# projection = nn.Linear(512, 128)  # Project to 128 dims

# # Combined encoder
# class vit_encoder_class(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.backbone = encoder_backbone
#         self.proj = projection

#     def forward(self, x):
#         # Normalize with ImageNet stats
#         mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device)
#         std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device)
#         x = (x - mean) / std
#         features = self.backbone(x).view(x.size(0), -1)
#         return self.proj(features)

# vit_encoder = vit_encoder_class().to(device).eval()
# print("✅ Using pre-trained ResNet18 encoder!")

In [9]:
expert_states_list = []
expert_actions_list = []
pot_start_list = []
pot_states_list1 = []
pot_states_list2 = []
image_latents_list0 = []
image_latents_list1 = []
for i in [2, 3]:
    for j in range(100):
        with open("rollouts/new_fixed_arm/rollout_seed%s_mode%s.pkl" % (j*10, i), "rb") as f:
            rollout = pkl.load(f)
            obs = rollout["observations"]
            actions = np.array(rollout["actions"])
            print(np.shape(actions))
            print("iteration" + str(j))
            pot1 = np.array(rollout["pot_states1"])
            pot2 = np.array(rollout["pot_states2"])
            pot = np.array(rollout["pot_start"])

            pot_start_list.append(np.concatenate((pot[0], pot[1])))

            T_target = 700

            if "camera_obs0" in rollout and "camera_obs1" in rollout:
                camera0_obs = np.array(rollout["camera_obs0"])  # (T, H, W, C)
                camera1_obs = np.array(rollout["camera_obs1"])

                # Pad or truncate images to T_target
                if camera0_obs.shape[0] > T_target:
                    camera0_obs = camera0_obs[:T_target]
                    camera1_obs = camera1_obs[:T_target]
                elif camera0_obs.shape[0] < T_target:
                    repeats_needed = T_target - camera0_obs.shape[0]
                    camera0_obs = np.vstack([camera0_obs, np.repeat(camera0_obs[-1][None], repeats_needed, axis=0)])
                    camera1_obs = np.vstack([camera1_obs, np.repeat(camera1_obs[-1][None], repeats_needed, axis=0)])

                camera0_latents = []
                camera1_latents = []

                for frame_idx in range(T_target):
                    img0 = torch.from_numpy(camera0_obs[frame_idx]).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
                    img1 = torch.from_numpy(camera1_obs[frame_idx]).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0

                    with torch.no_grad():
                        latents0 = encoder(img0)
                        latents1 = encoder(img1)

                    camera0_latents.append(latents0.cpu().numpy().squeeze())
                    camera1_latents.append(latents1.cpu().numpy().squeeze())

                camera0_latents = np.array(camera0_latents)
                camera1_latents = np.array(camera1_latents)
            else:
                print("Womp womp :(")

            robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
            robot0_eef_quat = np.array([o["robot0_eef_quat_site"] for o in obs])
            robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
            robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
            robot1_eef_quat = np.array([o["robot1_eef_quat_site"] for o in obs])
            robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

            repeats_needed = 700 - actions.shape[0]

            repeated_last = np.tile(actions[-1], (repeats_needed, 1))
            actions = np.vstack([actions, repeated_last])

            repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
            robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
            state = robot0_eef_pos

            repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
            robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
            robot0_eef_rotvec = R.from_quat(robot0_eef_quat).as_rotvec()
            state = np.hstack([state, robot0_eef_rotvec])


            repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
            robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
            robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
            state = np.hstack([state, robot0_gripper_pos])

            repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
            robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
            state = np.hstack([state, robot1_eef_pos])

            repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
            robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
            robot1_eef_rotvec = R.from_quat(robot1_eef_quat).as_rotvec()
            state = np.hstack([state, robot1_eef_rotvec])

            repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
            robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
            robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
            state = np.hstack([state, robot1_gripper_pos])

            repeated_last = np.tile(pot1[-1], (repeats_needed, 1))
            pot1 = np.vstack([pot1, repeated_last])
            repeated_last = np.tile(pot2[-1], (repeats_needed, 1))
            pot2 = np.vstack([pot2, repeated_last])
            pot_states_list1.append(pot1)
            pot_states_list2.append(pot2)

            # --- ENFORCE FINAL SHAPE FOR LATENTS --- #
            T_target = 700

            # Camera 0
            if camera0_latents.shape[0] > T_target:
                camera0_latents = camera0_latents[:T_target]
            elif camera0_latents.shape[0] < T_target:
                repeats_needed = T_target - camera0_latents.shape[0]
                last_frame = camera0_latents[-1:]
                padding = np.repeat(last_frame, repeats_needed, axis=0)
                camera0_latents = np.concatenate([camera0_latents, padding], axis=0)

            if camera1_latents.shape[0] > T_target:
                camera1_latents = camera1_latents[:T_target]
            elif camera1_latents.shape[0] < T_target:
                repeats_needed = T_target - camera1_latents.shape[0]
                last_frame = camera1_latents[-1:]
                padding = np.repeat(last_frame, repeats_needed, axis=0)
                camera1_latents = np.concatenate([camera1_latents, padding], axis=0)

            image_latents_list0.append(camera0_latents)
            image_latents_list1.append(camera1_latents)


            expert_states_list.append(state)
            expert_actions_list.append(actions)



expert_states_rotvec = np.stack(expert_states_list, axis=0)
expert_actions_rotvec = np.stack(expert_actions_list, axis=0)
pot_states_rotvec1 = np.stack(pot_states_list1, axis=0)
pot_states_rotvec2 = np.stack(pot_states_list2, axis=0)
pot_start_rotvec = np.stack(pot_start_list, axis=0)
image_latents_rotvec0 = np.stack(image_latents_list0, axis=0)
image_latents_rotvec1 = np.stack(image_latents_list1, axis=0)

(620, 14)
iteration0
(620, 14)
iteration1
(620, 14)
iteration2
(620, 14)
iteration3
(620, 14)
iteration4
(620, 14)
iteration5
(620, 14)
iteration6
(620, 14)
iteration7
(620, 14)
iteration8
(620, 14)
iteration9
(620, 14)
iteration10
(620, 14)
iteration11
(620, 14)
iteration12
(620, 14)
iteration13
(620, 14)
iteration14
(620, 14)
iteration15
(620, 14)
iteration16
(620, 14)
iteration17
(620, 14)
iteration18
(620, 14)
iteration19
(620, 14)
iteration20
(620, 14)
iteration21
(620, 14)
iteration22
(620, 14)
iteration23
(620, 14)
iteration24
(620, 14)
iteration25
(620, 14)
iteration26
(620, 14)
iteration27
(620, 14)
iteration28
(620, 14)
iteration29
(620, 14)
iteration30
(620, 14)
iteration31
(620, 14)
iteration32
(620, 14)
iteration33
(620, 14)
iteration34
(620, 14)
iteration35
(620, 14)
iteration36
(620, 14)
iteration37
(620, 14)
iteration38
(620, 14)
iteration39
(620, 14)
iteration40
(620, 14)
iteration41
(620, 14)
iteration42
(620, 14)
iteration43
(620, 14)
iteration44
(620, 14)
iteration4

In [11]:
print(np.shape(expert_states_rotvec))
print(np.shape(expert_actions_rotvec))
print(np.shape(pot_states_rotvec1))
print(np.shape(pot_states_rotvec2))
print(np.shape(pot_start_rotvec))
print(np.shape(image_latents_rotvec0))
print(np.shape(image_latents_rotvec1))

(200, 700, 14)
(200, 700, 14)
(200, 700, 3)
(200, 700, 3)
(200, 6)
(200, 700, 128)
(200, 700, 128)


In [11]:
D, T, C, H, W = image_latents_rotvec0.shape
image_latents_rotvec0 = image_latents_rotvec0.reshape(D, T, C*H*W)

D, T, C, H, W = image_latents_rotvec1.shape
image_latents_rotvec1 = image_latents_rotvec1.reshape(D, T, C*H*W)

ValueError: not enough values to unpack (expected 5, got 3)

In [12]:
print(np.shape(expert_states_rotvec))
print(np.shape(expert_actions_rotvec))
print(np.shape(pot_states_rotvec1))
print(np.shape(pot_states_rotvec2))
print(np.shape(pot_start_rotvec))
print(np.shape(image_latents_rotvec0))
print(np.shape(image_latents_rotvec1))

(200, 700, 14)
(200, 700, 14)
(200, 700, 3)
(200, 700, 3)
(200, 6)
(200, 700, 128)
(200, 700, 128)


In [13]:
import os
os.makedirs("TrainingDataDiffusion_fixed_arm", exist_ok=True)
np.save("TrainingDataDiffusion_fixed_arm/expert_states_newslower_20.npy", expert_states_rotvec)
np.save("TrainingDataDiffusion_fixed_arm/expert_actions_newslower_20.npy", expert_actions_rotvec)
np.save("TrainingDataDiffusion_fixed_arm/pot_states1_newslower_20.npy", pot_states_rotvec1)
np.save("TrainingDataDiffusion_fixed_arm/pot_states2_newslower_20.npy", pot_states_rotvec2)
np.save("TrainingDataDiffusion_fixed_arm/pot_start_newslower_20.npy", pot_start_rotvec)
np.save("TrainingDataDiffusion_fixed_arm/arm1_images_latents.npy", image_latents_rotvec0)
np.save("TrainingDataDiffusion_fixed_arm/arm2_images_latents.npy", image_latents_rotvec1)


In [11]:
import os
os.path.getsize("data/models/VAE_models_ICON/arm1_images_latents.npy")

17203328

In [ ]:
# with open("rollouts/rollout_seed0_mode2.pkl", "rb") as f:
#     rollout = pkl.load(f)
#     obs = rollout["observations"]
#     actions = np.array(rollout["actions"])

#     pos0 = actions[:,:3]
#     rotvec0 = actions[:,3:6]
#     gripper0 = actions[:,6]
#     pos1 = actions[:,7:10]
#     rotvec1 = actions[:,10:13]
#     gripper1 = actions[:,13]

#     rot6d_list0 = []
#     for rv in rotvec0:
#         rot6d_list0.append(rotvec_to_rot6d(rv))
#     rot6d0 = np.array(rot6d_list0)

#     rot6d_list1 = []
#     for rv in rotvec1:
#         rot6d_list1.append(rotvec_to_rot6d(rv))
#     rot6d1 = np.array(rot6d_list1)

#     actions = np.concatenate((pos0, rot6d0, gripper0.reshape(-1, 1), pos1, rot6d1, gripper1.reshape(-1, 1)), axis=1)

#     robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
#     robot0_eef_quat = np.array([o["robot0_eef_quat"] for o in obs])
#     robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
#     robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
#     robot1_eef_quat = np.array([o["robot1_eef_quat"] for o in obs])
#     robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

#     repeats_needed = 250 - actions.shape[0]

#     repeated_last = np.tile(actions[-1], (repeats_needed, 1))
#     actions = np.vstack([actions, repeated_last])

#     repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
#     robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
#     state = robot0_eef_pos

#     repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
#     robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
#     eef_rot6d0 = []
#     for q in robot0_eef_quat:
#         eef_rot6d0.append(quat_to_rot6d(q))
#     robot0_eef_rot6d = np.array(eef_rot6d0)
#     state = np.hstack([state, robot0_eef_rot6d])


#     repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
#     robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
#     robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
#     state = np.hstack([state, robot0_gripper_pos])

#     repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
#     robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
#     state = np.hstack([state, robot1_eef_pos])

#     repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
#     robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
#     eef_rot6d1 = []
#     for q in robot1_eef_quat:
#         eef_rot6d1.append(quat_to_rot6d(q))
#     robot1_eef_rot6d = np.array(eef_rot6d1)
#     state = np.hstack([state, robot1_eef_rot6d])

#     repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
#     robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
#     robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
#     state = np.hstack([state, robot1_gripper_pos])

# print(np.shape(state))
# print(np.shape(actions))

In [ ]:
expert_states_list = []
expert_actions_list = []
pot_states_list = []
for i in [2, 3]:
    for j in [0, 10, 20, 30, 40]:
        filepath = f"/content/drive/MyDrive/VAE_models_ICON/TrainingData/rollout_seed{j}_mode{i}.h5"

        with h5py.File(filepath, "r") as f:
            # obs = f["observations"]
            actions = np.array(f["actions"])
            pot = np.array(f["pot_pos"])

            pot_states_list.append(np.concatenate((pot[0], pot[1])))

            pos0 = actions[:,:3]
            rotvec0 = actions[:,3:6]
            gripper0 = actions[:,6]
            pos1 = actions[:,7:10]
            rotvec1 = actions[:,10:13]
            gripper1 = actions[:,13]

            rot6d_list0 = []
            for rv in rotvec0:
                rot6d_list0.append(rotvec_to_rot6d(rv))
            rot6d0 = np.array(rot6d_list0)

            rot6d_list1 = []
            for rv in rotvec1:
                rot6d_list1.append(rotvec_to_rot6d(rv))
            rot6d1 = np.array(rot6d_list1)

            actions = np.concatenate((pos0, rot6d0, gripper0.reshape(-1, 1), pos1, rot6d1, gripper1.reshape(-1, 1)), axis=1)

            robot0_eef_pos = np.array([o["robot0_eef_pos"] for o in obs])
            robot0_eef_quat = np.array([o["robot0_eef_quat_site"] for o in obs])
            robot0_gripper_pos = np.array([o["robot0_gripper_pos"] for o in obs])
            robot1_eef_pos = np.array([o["robot1_eef_pos"] for o in obs])
            robot1_eef_quat = np.array([o["robot1_eef_quat_site"] for o in obs])
            robot1_gripper_pos = np.array([o["robot1_gripper_pos"] for o in obs])

            repeats_needed = 400 - actions.shape[0]

            repeated_last = np.tile(actions[-1], (repeats_needed, 1))
            actions = np.vstack([actions, repeated_last])

            repeated_last = np.tile(robot0_eef_pos[-1], (repeats_needed, 1))
            robot0_eef_pos = np.vstack([robot0_eef_pos, repeated_last])
            state = robot0_eef_pos

            repeated_last = np.tile(robot0_eef_quat[-1], (repeats_needed, 1))
            robot0_eef_quat = np.vstack([robot0_eef_quat, repeated_last])
            eef_rot6d0 = []
            for q in robot0_eef_quat:
                eef_rot6d0.append(quat_to_rot6d(q))
            robot0_eef_rot6d = np.array(eef_rot6d0)
            state = np.hstack([state, robot0_eef_rot6d])


            repeated_last = np.tile(robot0_gripper_pos[-1], (repeats_needed, 1))
            robot0_gripper_pos = robot0_gripper_pos.reshape(-1, 1)
            robot0_gripper_pos = np.vstack([robot0_gripper_pos, repeated_last])
            state = np.hstack([state, robot0_gripper_pos])

            repeated_last = np.tile(robot1_eef_pos[-1], (repeats_needed, 1))
            robot1_eef_pos = np.vstack([robot1_eef_pos, repeated_last])
            state = np.hstack([state, robot1_eef_pos])

            repeated_last = np.tile(robot1_eef_quat[-1], (repeats_needed, 1))
            robot1_eef_quat = np.vstack([robot1_eef_quat, repeated_last])
            eef_rot6d1 = []
            for q in robot1_eef_quat:
                eef_rot6d1.append(quat_to_rot6d(q))
            robot1_eef_rot6d = np.array(eef_rot6d1)
            state = np.hstack([state, robot1_eef_rot6d])

            repeated_last = np.tile(robot1_gripper_pos[-1], (repeats_needed, 1))
            robot1_gripper_pos = robot1_gripper_pos.reshape(-1, 1)
            robot1_gripper_pos = np.vstack([robot1_gripper_pos, repeated_last])
            state = np.hstack([state, robot1_gripper_pos])

            expert_states_list.append(state)
            expert_actions_list.append(actions)

expert_states_rot6d = np.stack(expert_states_list, axis=0)
expert_actions_rot6d = np.stack(expert_actions_list, axis=0)
pot_states_rot6d = np.stack(pot_states_list, axis=0)

In [ ]:
print(np.shape(expert_states_rot6d))
print(np.shape(expert_actions_rot6d))
print(np.shape(pot_states_rot6d))

(20, 400, 20)
(20, 400, 20)
(20, 6)


In [ ]:
np.save("data/expert_states_rot6d_site_grippause_20.npy", expert_states_rot6d)
np.save("data/expert_actions_rot6d_site_grippause_20.npy", expert_actions_rot6d)
np.save("data/pot_states_rot6d_site_grippause_20.npy", pot_states_rot6d)